# Filter step-edge example

This notebook starts with the Circular Polynomial Gradient Filter (CPGF), but it is meant to be a tiny template for trying the other included filters too. It makes a `1024 x 1024` image where the left half is black and the right half is white, runs one filter, and plots the output.

Most included filters follow the same pattern. Choose a filter definition, choose the matching `ExecutionPath`, run `run_filter`, and look at `gradient_x` and `gradient_y`. To try another standard gradient filter, swap the import and the `definition = ...` line. Useful examples are `sobel_definition(kernel_size=3)`, `derivative_of_gaussian_definition(sigma=2.0)`, `haar_box_gradient_definition(radius=8)`, `robust_local_plane_gradient_definition(radius=3, weighting="huber")`, and `perona_malik_gradient_definition(iterations=10, kappa=0.2)`.

Angular filters are orientation-bank filters. They return one response image per angle, not just `gradient_x` and `gradient_y`. For those, use `anisotropic_gaussian_orientation_bank_definition(...)` with `run_orientation_bank(...)`. You can plot one angle with `result.responses[0, angle_index]`, or collapse the bank with `collapse_orientation_bank(...)` when you want gradient-like outputs.


## Setup

First import only what this CPGF example needs. `time` is only here so the notebook can print how long the filter took to run.

When you switch filters, this import line is one of the places to edit. For another standard gradient filter, import its definition function and keep `run_filter`. For an angular orientation-bank filter, import `anisotropic_gaussian_orientation_bank_definition`, `run_orientation_bank`, and `collapse_orientation_bank`.


In [ ]:
# Imports and local reloads.
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import ExecutionPath, cpgf_definition, run_filter

## Filter settings and test image

Set the image and filter parameters here. For CPGF, a larger `radius` uses a bigger neighborhood around each pixel, and a larger `degree` fits a higher-order polynomial.

The `path` should match the kind of filter you choose. Use `ExecutionPath.SEPARABLE` for Sobel, Scharr, Prewitt, Farid-Simoncelli, and derivative-of-Gaussian filters. Use `ExecutionPath.SPARSE_OFFSETS` for CPGF and sparse central differences, `ExecutionPath.BOX_INTEGRAL` for Haar box gradients, `ExecutionPath.RECURSIVE` for Deriche, `ExecutionPath.NONLINEAR_WINDOW` for robust local plane gradients, `ExecutionPath.ITERATIVE` for Perona-Malik, and `ExecutionPath.ORIENTATION_BANK` for angular filters.


In [ ]:
# Change the CPGF settings and build the test image.
radius = 2  # Wider radius samples more pixels around each point.
degree = 2  # Higher degree fits a higher-order polynomial surface.
path = ExecutionPath.SPARSE_OFFSETS  # Try SPATIAL_DENSE to compare paths.

size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

## Run the filter

Now build the CPGF definition and run it. The timer measures only the filter run, not the image setup above.

For another standard gradient filter, replace the first line in the next cell and leave the rest of the cell alone. For example, use `definition = sobel_definition(kernel_size=3)` with `path = ExecutionPath.SEPARABLE`, or use `definition = robust_local_plane_gradient_definition(radius=3, weighting="huber")` with `path = ExecutionPath.NONLINEAR_WINDOW`.

For an angular filter, this cell changes a little more because the output is a bank of directional responses. Build `definition = anisotropic_gaussian_orientation_bank_definition(angle_count=8, sigma_parallel=1.0, sigma_perpendicular=2.0)`, call `result = run_orientation_bank(definition, image, path=ExecutionPath.ORIENTATION_BANK, boundary=definition.default_boundary)`, and then either plot `result.responses` directly or collapse it before plotting.


In [ ]:
# Time the filter call.
definition = cpgf_definition(radius=radius, degree=degree)  # Build the CPGF weights.
start = time.perf_counter()
gradient_x, gradient_y = run_filter(
    definition,
    image,
    path=path,  # Use the execution path selected above.
    boundary=definition.default_boundary,  # Use the boundary mode chosen by CPGF.
)
elapsed = time.perf_counter() - start

print(f"{elapsed:.4f} seconds to run CPGF")

## Filter weights

This optional cell shows the filter weights. It works for filters with fixed spatial kernels, such as CPGF, Sobel, Scharr, Prewitt, Farid-Simoncelli, derivative-of-Gaussian, sparse central differences, and Haar box gradients.

Recursive, nonlinear, and iterative filters do not have one fixed pair of image kernels, so skip this cell for those. Angular orientation-bank filters store one kernel per angle; change `angle_index` to choose which angular kernel to inspect.


In [ ]:
# Plot the filter weights.
implementation = definition.require_implementation()
angle_index = 0

if implementation.orientation_kernels is None:
    weights = definition.dense_kernels()
    titles = ("kernel_x", "kernel_y")
else:
    weights = (implementation.orientation_kernels[angle_index],)
    titles = (f"angle {float(implementation.angles[angle_index]):.2f} rad",)

weight_limit = max(float(weight.abs().max()) for weight in weights)
fig, axes = plt.subplots(1, len(weights), figsize=(4 * len(weights), 3), squeeze=False)

for ax, weight, title in zip(axes[0], weights, titles, strict=False):
    ax.imshow(weight, cmap="gray", vmin=-weight_limit, vmax=weight_limit)
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(f"{definition.name} weights")
fig.tight_layout()

## Gradient results

The image changes left-to-right, so the main response should appear in `gradient_x`. The `gradient_y` output should be basically zero because the image does not change top-to-bottom.

Both plots use the same grayscale range. That matters here because plotting `gradient_y` by itself can stretch tiny roundoff values until they look like a real edge.

This plotting cell expects ordinary `gradient_x` and `gradient_y` tensors. If you are using an angular orientation bank, either set `gradient_x` and `gradient_y` from `collapse_orientation_bank(result, mode="least_squares_projection")`, or replace the plots with one directional response such as `result.responses[0, 0]`. The bank angles are stored in `result.angles`; angle `0` points along columns, and angle `pi / 2` points along rows.


In [ ]:
# Plot gradient_x and gradient_y.
limit = max(float(gradient_x.abs().max()), float(gradient_y.abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].imshow(gradient_x[0], cmap="gray", vmin=-limit, vmax=limit)
axes[0].set_title("gradient_x")

axes[1].imshow(gradient_y[0], cmap="gray", vmin=-limit, vmax=limit)
axes[1].set_title("gradient_y")

for ax in axes:
    ax.axis("off")

fig.suptitle(f"CPGF radius={radius}, degree={degree}, time={elapsed:.4f} s")
fig.tight_layout()